# AR(p) synthetic series vs TimesFM 2.5

**Goal:** Compare one-step-ahead forecasts from a fitted **AR(p)** model to **TimesFM** on synthetic **AR(p)** data.

**Setting (from the experiment plan):**
- **Horizon:** $t{+}1$ only.
- **Window:** First fit uses history $y_0,\ldots,y_{10}$ (length 11); forecast $y_{11}$. Then expand by one each time: history $y_0,\ldots,y_k$ predicts $y_{k+1}$ for $k=10,\ldots,n-2$. With $n=100$, forecast origins are $k+1 = 11,\ldots,99$ (**89** test points).
- **Metric:** mean squared error (MSE) of one-step errors.
- **DGP:** Gaussian AR(**p**) with **p ∈ {2, 5}**, **n = 100**. Classical model order matches the DGP (**ARIMA(p,0,0)**).

**Interpretation:** The AR model is *estimated* on each expanding history; TimesFM receives the same history as its *input prompt* (no training on labels).

## 1. Setup

**Hugging Face (no CLI needed):** downloads use the Hub. Set a **read** token as `HF_TOKEN` ([tokens](https://huggingface.co/settings/tokens)). One-time setup:
- **Dependency:** from the repo root, `pip install -e ".[experiments]"` (installs `python-dotenv` for `.env` loading).
- **Env file:** copy `.env.example` to `.env` in the repo root, then add your token on the `HF_TOKEN=` line.
- **Or** export in the terminal before Jupyter: `export HF_TOKEN=hf_...`

The first code cell calls `load_dotenv(REPO / ".env")` so the repo-root `.env` is loaded automatically.

Imports, repo path (so `timesfm` resolves), and fixed **random seed** for reproducibility.

In [15]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import warnings

warnings.filterwarnings("ignore", category=UserWarning, module="statsmodels")

# Repo root: notebook lives in experiments_nikhilesh_experiments/
HERE = Path.cwd().resolve()
REPO = HERE if (HERE / "src" / "timesfm").is_dir() else HERE.parent
try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass
sys.path.insert(0, str(REPO / "src"))

import timesfm
from statsmodels.tsa.arima.model import ARIMA

RNG_SEED = 42
N = 100
K_FIRST = 11
rng = np.random.default_rng(RNG_SEED)

## 2. Simulate AR(p) data

Simple recursion: $y_t = \sum_{i=1}^p \phi_i y_{t-i} + \varepsilon_t$. Coefficients are fixed and chosen so the process is stable (roughly stationary).

In [16]:
def simulate_ar(p: int, n: int, rng: np.random.Generator) -> np.ndarray:
    if p == 2:
        phi = np.array([0.45, 0.35])
    elif p == 5:
        phi = np.array([0.20, 0.15, 0.12, 0.10, 0.08])
    else:
        raise ValueError("p must be 2 or 5")
    eps = rng.normal(0.0, 1.0, n)
    y = np.zeros(n, dtype=np.float64)
    y[:p] = eps[:p]
    for t in range(p, n):
        # y_t = phi_1 * y_{t-1} + ... + phi_p * y_{t-p} + noise
        ar_sum = 0.0
        for lag in range(1, p + 1):
            ar_sum += phi[lag - 1] * y[t - lag]
        y[t] = ar_sum + eps[t]
    return y


y_ar2 = simulate_ar(2, N, rng)
y_ar5 = simulate_ar(5, N, rng)

## 3. One-step AR forecast (classical)

Fit **ARIMA(p,0,0)** on `history` and return the one-step forecast. Returns `None` if estimation fails (rare on short windows).

In [17]:
def forecast_ar(history: np.ndarray, p: int) -> float | None:
    try:
        if len(history) <= p:
            return None
        fit = ARIMA(history.astype(np.float64), order=(p, 0, 0)).fit()
        return float(np.asarray(fit.forecast(steps=1)).ravel()[0])
    except Exception:
        raise Exception 

## 4. Load TimesFM (once)

Same model configuration as other experiments in this folder; **horizon = 1** at evaluation time.

In [18]:
torch.set_float32_matmul_precision("high")
tfm = timesfm.TimesFM_2p5_200M_torch.from_pretrained(
    "google/timesfm-2.5-200m-pytorch",
    torch_compile=False,
)
tfm.compile(
    timesfm.ForecastConfig(
        max_context=1024,
        max_horizon=256,
        normalize_inputs=True,
        use_continuous_quantile_head=True,
        force_flip_invariance=True,
        infer_is_positive=False,
        fix_quantile_crossing=True,
    )
)

## 5. Expanding-window loop and MSE

For each DGP order **p**, run the test indices $k = 11,\ldots,99$: actual $= y[k]$, history $= y[:k]$. Compare squared errors for AR vs TimesFM.

In [19]:
rows: list[dict] = []
test_ks = list(range(K_FIRST, N))

for p, y in [(2, y_ar2), (5, y_ar5)]:
    se_ar: list[float] = []
    se_tf: list[float] = []
    n_fail = 0
    for k in test_ks:
        actual = float(y[k])
        hist = y[:k].astype(np.float32)
        pred_tf = float(tfm.forecast(horizon=1, inputs=[hist])[0][0, 0])
        pred_ar = forecast_ar(y[:k], p)
        e_ar = actual - pred_ar
        e_tf = actual - pred_tf
        se_ar.append(e_ar**2)
        se_tf.append(e_tf**2)
    rows.append(
        {
            "p_dgp": p,
            "n_test": len(test_ks),
            "ar_fit_failures": n_fail,
            "mse_ar": float(np.mean(se_ar)) if se_ar else float("nan"),
            "mse_timesfm": float(np.mean(se_tf)) if se_tf else float("nan"),
        }
    )

results = pd.DataFrame(rows)
results

,p_dgp,n_test,ar_fit_failures,mse_ar,mse_timesfm
0,2,89,0,0.631848,0.655677
1,5,89,0,1.117969,0.984800


## 6. Save table

CSV under `output/` next to this notebook.

In [20]:
out_dir = HERE / "output"
out_dir.mkdir(parents=True, exist_ok=True)
path = out_dir / "ar_vs_timesfm_results.csv"
results.to_csv(path, index=False)
print(path)

/Users/nikhileshbelulkar/Documents/timesfm_google/experiments_nikhilesh_experiments/output/ar_vs_timesfm_results.csv
